# 04 — Weekly Team Strength

This notebook updates team strength sequentially using the new in season evidence created in `03_Inseason_Feature_Engineering.ipynb`.

The original preseason model remains unchanged.

The weekly model follows a sequential principle:

> The previous week's team strength is the prior. The most recently completed week provides new evidence.

For each target week, this notebook:

- Loads the frozen preseason ratings for reference
- Loads the previous week's team strength state
- Loads the target week's new in season feature table
- Applies the opponent adjusted weekly strength update
- Preserves separate offensive and defensive weekly signals
- Calculates current weekly rankings
- Compares rank and strength movement to the previous week
- Saves a separate weekly team strength file

No original preseason file or prior weekly file is overwritten.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the output from:

`03_Inseason_Feature_Engineering.ipynb`

For Week 3:

- `TARGET_WEEK = 3`
- `PRIOR_WEEK = 2`

The Week 2 team strength file becomes the entering prior for Week 3.

In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 11
PRIOR_WEEK = TARGET_WEEK - 1

PRESEASON_STRENGTH_PATH = (
    PROCESSED_DIR
    / "2026_team_strength.parquet"
)

PRIOR_WEEK_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

INSEASON_FEATURES_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

print("Preseason ratings:", PRESEASON_STRENGTH_PATH)
print("Prior-week ratings:", PRIOR_WEEK_STRENGTH_PATH)
print("New in-season features:", INSEASON_FEATURES_PATH)
print("Weekly output:", OUTPUT_PATH)

Preseason ratings: ..\..\data\processed\2026_team_strength.parquet
Prior-week ratings: ..\..\data\processed\weekly\week_10_sequential_team_strength.parquet
New in-season features: ..\..\data\processed\weekly\week_11_inseason_features.parquet
Weekly output: ..\..\data\processed\weekly\week_11_sequential_team_strength.parquet


# Load Frozen Preseason Team Strength

The preseason team strength file is the final output of the original `05_Team_Strength_Model.ipynb`.

Its `team_strength` value is already expressed in point differential terms, so it can serve directly as the prior for weekly updates.

We immediately rename the important preseason columns so it is impossible to confuse the original ratings with the new weekly ratings.


In [3]:
preseason = pd.read_parquet(
    PRESEASON_STRENGTH_PATH
).copy()

required_preseason_columns = [
    "team",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

missing_preseason = [
    column
    for column in required_preseason_columns
    if column not in preseason.columns
]

if missing_preseason:
    raise KeyError(
        "Missing required preseason columns: "
        + ", ".join(missing_preseason)
    )

preseason = preseason.rename(
    columns={
        "strength_rank": "preseason_strength_rank",
        "team_strength": "preseason_team_strength",
        "baseline_team_strength": "preseason_baseline_team_strength",
        "personnel_adjustment": "preseason_personnel_adjustment",
        "personnel_strength": "preseason_personnel_strength",
        "roster_continuity": "preseason_roster_continuity",
        "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
    }
)

print("Preseason teams:", len(preseason))
print("Unique teams:", preseason["team"].nunique())

display(
    preseason[
        [
            "preseason_strength_rank",
            "team",
            "preseason_team_strength",
            "preseason_baseline_team_strength",
            "preseason_personnel_adjustment",
            "preseason_roster_continuity_adjustment"
        ]
    ].sort_values(
        "preseason_strength_rank"
    )
)


Preseason teams: 32
Unique teams: 32


,preseason_strength_rank,team,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.260091
1,2,LA,4.762452,3.257133,2.100811,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.037533
4,5,DEN,3.569507,2.355154,0.979753,1.253003
5,6,BAL,3.074111,2.710864,0.636254,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.576742
8,9,PHI,2.451919,2.324982,0.264690,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.290631


# Load Prior-Week Team Strength

The previous week's team strength file is the starting point for the new weekly update.

For Week 3, the model begins with the Week 2 ratings.

This prevents older games from being counted repeatedly and creates a true sequential rating system.

In [4]:
prior_week = pd.read_parquet(
    PRIOR_WEEK_STRENGTH_PATH
).copy()

required_prior_columns = [
    "team",
    "weekly_strength_rank",
    "weekly_team_strength",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment"
]

missing_prior = [
    column
    for column in required_prior_columns
    if column not in prior_week.columns
]

if missing_prior:
    raise KeyError(
        "Missing required prior-week columns: "
        + ", ".join(missing_prior)
    )

prior_week = prior_week[
    required_prior_columns
].rename(
    columns={
        "weekly_strength_rank":
            "prior_week_strength_rank",
        "weekly_team_strength":
            "prior_week_team_strength",
        "cumulative_offense_adjustment":
            "prior_cumulative_offense_adjustment",
        "cumulative_defense_adjustment":
            "prior_cumulative_defense_adjustment"
    }
)

print(
    f"Week {PRIOR_WEEK} teams:",
    len(prior_week)
)

print(
    "Unique teams:",
    prior_week["team"].nunique()
)

display(
    prior_week.sort_values(
        "prior_week_strength_rank"
    )
)

Week 10 teams: 32
Unique teams: 32


,team,prior_week_strength_rank,prior_week_team_strength,prior_cumulative_offense_adjustment,prior_cumulative_defense_adjustment
0,BUF,1,7.483841,11.213581,-0.043026
1,JAX,2,5.875962,6.823763,9.517156
2,SF,3,4.990056,7.657233,3.975626
3,LA,4,4.643452,0.930460,3.645281
4,SEA,5,4.207138,-0.488294,6.876527
5,DEN,6,3.953856,-0.891564,3.473258
6,KC,7,3.818115,2.829545,5.784367
7,DET,8,3.609975,5.246846,-0.974047
8,CHI,9,3.472897,5.291976,3.506798
9,GB,10,3.446027,1.215419,2.130240


# Load New Weekly In-Season Features

These features contain the new evidence generated in notebook `03`.

For Week 3, they describe each team's Week 2 performance relative to its entering Week 2 expectation.

The primary values used here are:

- `weekly_strength_update`
- `weekly_offense_adjustment`
- `weekly_defense_adjustment`

The overall strength update is opponent adjusted and uses diminishing returns for extreme single game results.

In [5]:
inseason = pd.read_parquet(
    INSEASON_FEATURES_PATH
).copy()

required_inseason_columns = [
    "team",
    "games_in_update",
    "entering_team_strength",
    "entering_strength_rank",
    "average_opponent_strength",
    "average_opponent_rank",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

missing_inseason = [
    column
    for column in required_inseason_columns
    if column not in inseason.columns
]

if missing_inseason:
    raise KeyError(
        "Missing required in-season columns: "
        + ", ".join(missing_inseason)
    )

print("In-season teams:", len(inseason))
print(
    "Unique teams:",
    inseason["team"].nunique()
)

display(
    inseason[
        required_inseason_columns
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

In-season teams: 28
Unique teams: 28


,team,games_in_update,entering_team_strength,entering_strength_rank,average_opponent_strength,average_opponent_rank,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
18,MIN,1,1.673649,14,3.446027,10.0,20.0,-3.272378,23.272378,1.395792,1.002440,2.256012
8,DET,1,3.609975,8,1.490937,15.0,25.0,3.619038,21.380962,1.364919,1.679124,2.182696
14,LA,1,4.643452,4,-4.795138,29.0,28.0,7.938590,20.061410,1.338407,1.293397,2.546968
5,CIN,1,2.995699,11,2.844367,12.0,20.0,1.651332,18.348668,1.296648,2.179878,1.033449
13,KC,1,3.818115,7,-2.831625,22.0,22.0,5.149740,16.850260,1.252120,1.517028,1.570600
6,CLE,1,-4.106187,25,2.325586,13.0,3.0,-4.931773,7.931773,0.769237,0.510424,0.113995
15,LAC,1,-4.739916,28,-1.682630,19.0,3.0,-4.557286,7.557286,0.739262,-0.239884,0.563687
3,BUF,1,7.483841,1,-6.315289,31.0,19.0,12.299130,6.700870,0.667723,0.786391,1.589962
12,JAX,1,5.875962,2,-6.584857,32.0,17.0,10.960818,6.039182,0.609699,2.126282,-0.070146
4,CAR,1,-3.664380,24,-2.781537,21.0,3.0,-2.382843,5.382843,0.549899,-0.731093,0.972478


# Merge Prior Week State with New Evidence

The weekly model now combines three pieces of information:

1. Frozen preseason information for long term reference
2. Previous week's team strength as the current prior
3. Most recent week's opponent adjusted performance as new evidence

All 32 teams must remain in the table.

In [6]:
weekly_strength = (
    preseason
    .merge(
        prior_week,
        on="team",
        how="left",
        validate="one_to_one"
    )
    .merge(
        inseason,
        on="team",
        how="left",
        validate="one_to_one"
    )
)

weekly_fill_columns = [
    "games_in_update",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

weekly_strength[
    weekly_fill_columns
] = (
    weekly_strength[
        weekly_fill_columns
    ]
    .fillna(0.0)
)

print("Merged teams:", len(weekly_strength))
print(
    "Unique teams:",
    weekly_strength["team"].nunique()
)

print(
    "Missing prior-week strengths:",
    weekly_strength[
        "prior_week_team_strength"
    ].isna().sum()
)

Merged teams: 32
Unique teams: 32
Missing prior-week strengths: 0


# Sequential Weekly Rating Update

The previous week's team strength is the starting point.

The current week's opponent adjusted performance update is then applied:

`weekly_team_strength = prior_week_team_strength + weekly_strength_update`

Therefore, for Week 3:

`Week 3 Strength = Week 2 Strength + Week 2 Performance Update`

Older results remain embedded in the prior week rating and are not counted again.

The offensive and defensive weekly adjustments remain available separately for the weekly scoring model.

In [7]:
weekly_strength["team_strength_change"] = (
    weekly_strength["weekly_strength_update"]
)

weekly_strength["weekly_team_strength"] = (
    weekly_strength["prior_week_team_strength"]
    + weekly_strength["team_strength_change"]
)

display(
    weekly_strength[
        [
            "team",
            "prior_week_team_strength",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment",
            "team_strength_change",
            "weekly_team_strength"
        ]
    ].sort_values(
        "weekly_team_strength",
        ascending=False
    )
)

,team,prior_week_team_strength,weekly_offense_adjustment,weekly_defense_adjustment,team_strength_change,weekly_team_strength
0,BUF,7.483841,0.786391,1.589962,0.667723,8.151564
11,JAX,5.875962,2.126282,-0.070146,0.609699,6.485660
1,LA,4.643452,1.293397,2.546968,1.338407,5.981859
9,SF,4.990056,3.256209,-2.390219,0.218765,5.208822
12,KC,3.818115,1.517028,1.570600,1.252120,5.070234
3,DET,3.609975,1.679124,2.182696,1.364919,4.974895
2,SEA,4.207138,-0.639183,1.514388,0.102937,4.310075
20,CIN,2.995699,2.179878,1.033449,1.296648,4.292347
4,DEN,3.953856,0.000000,0.000000,0.000000,3.953856
18,CHI,3.472897,0.000000,0.000000,0.000000,3.472897


In [8]:
weekly_strength["cumulative_offense_adjustment"] = (
    weekly_strength["prior_cumulative_offense_adjustment"]
    + weekly_strength["weekly_offense_adjustment"]
)

weekly_strength["cumulative_defense_adjustment"] = (
    weekly_strength["prior_cumulative_defense_adjustment"]
    + weekly_strength["weekly_defense_adjustment"]
)

assert weekly_strength[
    "cumulative_offense_adjustment"
].notna().all()

assert weekly_strength[
    "cumulative_defense_adjustment"
].notna().all()

# Updated Weekly Rankings

The current weekly rank is based on `weekly_team_strength`.

Ranking movement is now measured relative to the immediately previous week rather than the preseason ranking.

A positive `rank_change` means the team moved up.

For example:

- Week 2 rank = 8
- Week 3 rank = 5
- Rank change = +3

In [9]:
weekly_strength = weekly_strength.sort_values(
    [
        "weekly_team_strength",
        "prior_week_team_strength"
    ],
    ascending=[False, False]
).reset_index(drop=True)

weekly_strength["weekly_strength_rank"] = (
    np.arange(
        1,
        len(weekly_strength) + 1
    )
)

weekly_strength["rank_change"] = (
    weekly_strength["prior_week_strength_rank"]
    - weekly_strength["weekly_strength_rank"]
)

weekly_comparison_table = (
    weekly_strength[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
    .copy()
)

display(weekly_comparison_table)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change
0,1,BUF,8.151564,1,7.483841,0.667723,0
1,2,JAX,6.485660,2,5.875962,0.609699,0
2,3,LA,5.981859,4,4.643452,1.338407,1
3,4,SF,5.208822,3,4.990056,0.218765,-1
4,5,KC,5.070234,7,3.818115,1.252120,2
5,6,DET,4.974895,8,3.609975,1.364919,2
6,7,SEA,4.310075,5,4.207138,0.102937,-2
7,8,CIN,4.292347,11,2.995699,1.296648,3
8,9,DEN,3.953856,6,3.953856,0.000000,-3
9,10,CHI,3.472897,9,3.472897,0.000000,-1


# Biggest Weekly Movers

This table is diagnostic only.

It shows which teams experienced the largest overall rating changes from the previous week to the current week.

The underlying projection model still retains the full preseason, offense, defense, opponent and weekly information.

In [10]:
mover_columns = [
    "team",
    "prior_week_team_strength",
    "team_strength_change",
    "weekly_team_strength",
    "prior_week_strength_rank",
    "weekly_strength_rank",
    "rank_change"
]

biggest_risers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=False
    )
    .head(10)
)

biggest_fallers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=True
    )
    .head(10)
)

print("BIGGEST RISERS")
display(biggest_risers)

print()
print("BIGGEST FALLERS")
display(biggest_fallers)

BIGGEST RISERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
10,MIN,1.673649,1.395792,3.069441,14,11,3
5,DET,3.609975,1.364919,4.974895,8,6,2
2,LA,4.643452,1.338407,5.981859,4,3,1
7,CIN,2.995699,1.296648,4.292347,11,8,3
4,KC,3.818115,1.252120,5.070234,7,5,2
23,CLE,-4.106187,0.769237,-3.336949,25,24,1
24,LAC,-4.739916,0.739262,-4.000654,28,25,3
0,BUF,7.483841,0.667723,8.151564,1,1,0
1,JAX,5.875962,0.609699,6.485660,2,2,0
20,CAR,-3.664380,0.549899,-3.114481,24,21,3



BIGGEST FALLERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
11,GB,3.446027,-1.395792,2.050235,10,12,-2
15,NE,1.490937,-1.364919,0.126018,15,16,-1
29,ARI,-4.795138,-1.338407,-6.133545,29,30,-1
13,PIT,2.844367,-1.296648,1.547719,12,14,-2
25,ATL,-2.831625,-1.252120,-4.083745,22,26,-4
12,HOU,2.325586,-0.769237,1.556349,13,13,0
19,BAL,-1.682630,-0.739262,-2.421892,19,20,-1
30,NYJ,-6.315289,-0.667723,-6.983012,31,31,0
31,TEN,-6.584857,-0.609699,-7.194555,32,32,0
22,NO,-2.781537,-0.549899,-3.331436,21,23,-2


# Update Diagnostics

These diagnostics check the size and distribution of the sequential weekly update.

The comparison is now:

`Prior Week Strength → Current Week Strength`

rather than:

`Preseason Strength → Current Week Strength`

In [11]:
movement = (
    weekly_strength[
        "team_strength_change"
    ].abs()
)

print(
    f"WEEK {PRIOR_WEEK} → "
    f"WEEK {TARGET_WEEK} TEAM-STRENGTH MOVEMENT"
)

print(
    weekly_strength[
        "team_strength_change"
    ]
    .describe()
    .round(3)
)

print()

print(
    "Mean absolute movement:",
    round(
        movement.mean(),
        3
    )
)

print(
    "Maximum absolute movement:",
    round(
        movement.max(),
        3
    )
)

print(
    "Team with largest movement:",
    weekly_strength.loc[
        movement.idxmax(),
        "team"
    ]
)

print()
print("RATING DISTRIBUTIONS")

distribution_check = pd.DataFrame(
    {
        "metric": [
            f"Week {PRIOR_WEEK} Team Strength",
            f"Week {TARGET_WEEK} Team Strength"
        ],
        "mean": [
            weekly_strength[
                "prior_week_team_strength"
            ].mean(),
            weekly_strength[
                "weekly_team_strength"
            ].mean()
        ],
        "std": [
            weekly_strength[
                "prior_week_team_strength"
            ].std(),
            weekly_strength[
                "weekly_team_strength"
            ].std()
        ],
        "min": [
            weekly_strength[
                "prior_week_team_strength"
            ].min(),
            weekly_strength[
                "weekly_team_strength"
            ].min()
        ],
        "max": [
            weekly_strength[
                "prior_week_team_strength"
            ].max(),
            weekly_strength[
                "weekly_team_strength"
            ].max()
        ]
    }
)

display(
    distribution_check.round(3)
)

WEEK 10 → WEEK 11 TEAM-STRENGTH MOVEMENT
count    32.000
mean     -0.000
std       0.859
min      -1.396
25%      -0.624
50%       0.000
75%       0.624
max       1.396
Name: team_strength_change, dtype: float64

Mean absolute movement: 0.684
Maximum absolute movement: 1.396
Team with largest movement: MIN

RATING DISTRIBUTIONS


,metric,mean,std,min,max
0,Week 10 Team Strength,0.002,4.037,-6.585,7.484
1,Week 11 Team Strength,0.002,4.373,-7.195,8.152


# Sanity Checks

These checks verify the sequential weekly update.

The weekly file should:

- contain exactly 32 unique teams
- preserve the frozen preseason information
- contain a valid prior week rating for every team
- contain no missing current weekly strength values
- have `team_strength_change` equal to the new opponent adjusted weekly update
- calculate current strength as prior week strength plus the weekly update
- never overwrite the preseason or prior week files

In [12]:
assert len(weekly_strength) == 32, (
    "Expected 32 teams in weekly strength table."
)

assert (
    weekly_strength["team"].nunique()
    == 32
), (
    "Expected 32 unique teams."
)

assert weekly_strength[
    "preseason_team_strength"
].notna().all(), (
    "Missing frozen preseason ratings."
)

assert weekly_strength[
    "prior_week_team_strength"
].notna().all(), (
    "Missing prior-week ratings."
)

assert weekly_strength[
    "weekly_team_strength"
].notna().all(), (
    "Missing current weekly ratings."
)

assert np.allclose(
    weekly_strength[
        "team_strength_change"
    ],
    weekly_strength[
        "weekly_strength_update"
    ]
), (
    "Team-strength change does not match "
    "the opponent-adjusted weekly update."
)

assert np.allclose(
    weekly_strength[
        "weekly_team_strength"
    ],
    (
        weekly_strength[
            "prior_week_team_strength"
        ]
        + weekly_strength[
            "team_strength_change"
        ]
    )
), (
    "Sequential weekly strength formula "
    "is inconsistent."
)

played_mask = weekly_strength["games_in_update"] > 0

assert np.allclose(
    weekly_strength.loc[
        played_mask, "entering_team_strength"
    ],
    weekly_strength.loc[
        played_mask, "prior_week_team_strength"
    ]
), (
    "Notebook 03 entering strength does not match "
    "the prior-week team strength."
)

bye_mask = ~played_mask

assert np.allclose(
    weekly_strength.loc[
        bye_mask, "weekly_team_strength"
    ],
    weekly_strength.loc[
        bye_mask, "prior_week_team_strength"
    ]
), "Bye-week teams did not retain their previous strength."

print(
    "All sequential weekly team-strength "
    "sanity checks passed."
)

All sequential weekly team-strength sanity checks passed.


# Final Weekly Team Strength Table

The saved file contains the full state needed by the weekly projection pipeline.

This includes:

- current weekly strength and rank
- previous week's strength and rank
- week to week movement
- frozen preseason information
- opponent adjusted performance information
- separate offensive and defensive weekly signals

The compact ranking table shown earlier is only for interpretation. The saved dataset retains the additional model information.

In [13]:
final_columns = [
    "weekly_strength_rank",
    "team",
    "weekly_team_strength",

    "prior_week_strength_rank",
    "prior_week_team_strength",
    "team_strength_change",
    "rank_change",

    "games_in_update",
    "average_opponent_rank",
    "average_opponent_strength",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",

    "weekly_offense_adjustment",
    "weekly_defense_adjustment",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment",

    "preseason_strength_rank",
    "preseason_team_strength",
    "preseason_baseline_team_strength",
    "preseason_personnel_adjustment",
    "preseason_personnel_strength",
    "preseason_roster_continuity",
    "preseason_roster_continuity_adjustment"
]

weekly_team_strength = (
    weekly_strength[
        final_columns
    ]
    .copy()
)

display(
    weekly_team_strength
)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,games_in_update,average_opponent_rank,average_opponent_strength,...,weekly_defense_adjustment,cumulative_offense_adjustment,cumulative_defense_adjustment,preseason_strength_rank,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_personnel_strength,preseason_roster_continuity,preseason_roster_continuity_adjustment
0,1,BUF,8.151564,1,7.483841,0.667723,0,1.0,31.0,-6.315289,...,1.589962,11.999972,1.546936,1,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,JAX,6.485660,2,5.875962,0.609699,0,1.0,32.0,-6.584857,...,-0.070146,8.950045,9.447010,12,1.713136,1.746953,-0.223729,-0.149153,0.563830,0.200841
2,3,LA,5.981859,4,4.643452,1.338407,1,1.0,29.0,-4.795138,...,2.546968,2.223856,6.192249,2,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
3,4,SF,5.208822,3,4.990056,0.218765,-1,1.0,17.0,-0.453468,...,-2.390219,10.913442,1.585406,10,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631
4,5,KC,5.070234,7,3.818115,1.252120,2,1.0,22.0,-2.831625,...,1.570600,4.346574,7.354966,13,1.302244,1.484935,-0.187743,-0.125162,0.510870,-0.140090
5,6,DET,4.974895,8,3.609975,1.364919,2,1.0,15.0,1.490937,...,2.182696,6.925970,1.208649,4,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
6,7,SEA,4.310075,5,4.207138,0.102937,-2,1.0,27.0,-4.330605,...,1.514388,-1.127478,8.390915,3,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
7,8,CIN,4.292347,11,2.995699,1.296648,3,1.0,12.0,2.844367,...,1.033449,10.288631,2.321309,21,-0.739434,-1.268642,0.286293,0.190862,0.643678,0.714864
8,9,DEN,3.953856,6,3.953856,0.000000,-3,0.0,NaN,NaN,...,0.000000,-0.891564,3.473258,5,3.569507,2.355154,0.979753,0.653169,0.727273,1.253003
9,10,CHI,3.472897,9,3.472897,0.000000,-1,0.0,NaN,NaN,...,0.000000,5.291976,3.506798,19,-0.051934,-0.069709,0.002783,0.001856,0.537634,0.032209


# Save Weekly Team Strength

This writes only to:

`data/processed/weekly/`

The original:

`data/processed/2026_team_strength.parquet`

remains untouched.


In [14]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly_team_strength.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print()
print(
    "Original preseason file remains:",
    PRESEASON_STRENGTH_PATH
)


Saved: ..\..\data\processed\weekly\week_11_sequential_team_strength.parquet

Original preseason file remains: ..\..\data\processed\2026_team_strength.parquet


# What This Means for Week 3

At the end of this notebook, every team has:

- its frozen preseason rating
- its Week 2 entering rating and rank
- its opponent adjusted Week 2 performance
- its Week 2 offensive and defensive signals
- its Week 2 → Week 3 rating movement
- its updated Week 3 team strength rating
- its updated Week 3 rank

The next stage will apply the Week 3 injury update and then use the resulting team state to generate Week 3 game projections.

The weekly ranking table is diagnostic only.

The saved weekly team strength file retains the additional information required by the projection pipeline.